In [11]:
from functions import *
from firedrake import *
import numpy as np
from IPython.display import display, HTML
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection #need for quad mesh
import gmsh #unstructed mesh
from firedrake.pyplot import triplot #plotting the mesh
import pandas as pd
import logging
opts = {
    "eps_gen_non_hermitian": None,
    "eps_target": 0.95,              # off 1.0, which is a genuine eigenvalue
    "eps_target_magnitude": None,
    "st_type": "sinvert",
    "st_pc_type": "lu",
    "st_pc_factor_mat_solver_type": "mumps",
    "st_mat_mumps_icntl_14": 500,
}      

In [12]:
def FEEC_B_N1_2d1k(mesh, degree=1):
    """
    Builds the Mixed Function Space for the Rotated B-formulation.
    Flux (sigma) is in H(div) using Raviart-Thomas elements.
    Scalar (v) is in L^2 using Discontinuous Galerkin elements.
    """
    Sigma = FunctionSpace(mesh, "RT", degree)
    V = FunctionSpace(mesh, "DG", degree - 1)
    
    return Sigma * V

def B_formulation_2d1k(W,c, u=None, eps=Constant(1.0)):
    """
    Assembles the stabilized left-hand operator Ac and right-hand mass matrix M
    for the mixed scalar advection-diffusion problem.
    """
    sigma, v = TrialFunctions(W)
    tau, w = TestFunctions(W)
    
    # Define the spectral shift parameter c.
    # Mathematically, this must be > ||u||^2 / (2*eps)
    c = Constant(50.0) 
    
    # 1. Base principal saddle-point blocks
    a_base = (1.0 / eps) * inner(sigma, tau) * dx \
           + v * div(tau) * dx \
           - div(sigma) * w * dx
           
    # 2. Advective Perturbation
    if u is not None:
        a_adv = (1.0 / eps) * inner(v * u, tau) * dx
    else:
        a_adv = 0
        
    # 3. Spectral Shift (Stabilizing the bottom-right block)
    a_shift = c * v * w * dx
    
    # Full stabilized left-hand operator
    a = a_base + a_adv + a_shift
    
    # Right-hand mass matrix (only the v, w block)
    m = v * w * dx
    
    return a, m
    
def bcs_B_2d1k(W):
    """
    Returns an empty list because the primal Dirichlet condition (v=0) 
    is a natural boundary condition in the H(div) mixed formulation.
    """
    return []

def wind_func_constant(mesh):
    return as_vector([Constant(1.0), Constant(1.0)])
def wind0(mesh):
    return None

In [16]:
mesh = SquareMesh(8, 8, np.pi, quadrilateral=False, diagonal='crossed')
W =  FEEC_B_N1_2d1k(mesh, degree=1)
a, m = B_formulation_2d1k(W,c=10, u=None, eps=Constant(1.0))
bcs = bcs_B_2d1k(W)

In [17]:
# Setup and run the eigensolver using the Firedrake wrappers
eigensolver = make_eigensolver(a, m, bcs, n_evals=32, opts=opts)
shifted_eigs, eigfs = solve_eigenpairs(eigensolver, drop_harmonic=True)

# Recover the physical spectrum (assuming c = 50.0 in the form builder)
c_val = 10
physical_eigs = [val - c_val for val in shifted_eigs]
print(physical_eigs)

[(41.9914176506809+0j), (44.974878611334+0j), (44.974878611343385+0j), (47.861901906586866+0j), (49.985787514351806+0j), (49.98578751435182+0j), (52.71195161102295+0j), (52.71195161102342+0j), (57.0706818575032+0j), (57.07068185750508+0j), (57.293141842610126+0j), (59.5715591442701+0j), (59.57155914446746+0j), (63.774904734055085+0j), (63.77490473405548+0j), (66.26574273946433+0j), (66.26574273946436+0j), (68.47784161955983+0j), (68.4778416195609+0j), (69.72255874893011+0j), (72.19411989435625+0j), (72.19411989435653+0j), (77.44678849755093+0j), (77.44678849755425+0j), (77.5331464442132+0j), (77.5331464443049+0j), (79.39909915083875+0j), (79.39909915084043+0j), (82.52579249851064+0j), (82.5257924985697+0j), (84.25333272607418+0j), (86.92366152025932+0j), (86.92366152028353+0j), (90.4388789058723+0j)]


In [ ]:
p